In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from pydantic import BaseModel, Field
import pandas as pd

In [2]:
load_dotenv()

True

In [3]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length = 3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

In [13]:
client = OpenAI()
def analyse_note(note_text):
    response = client.responses.parse(
        model="gpt-5-nano",
        input=[
            {"role": "system", "content": "You are analysing the follow up notes of doctor's missed appointment. Provide output as defined in text format do not invent the data"},
            {"role": "user", "content": note_text},
        ],
        text_format=FollowupInsight,
    )
    parsed = response.output_parsed
    if parsed is None:  # model refused or returned nothing parseable
        raise ValueError("No parsed output returned")
    return parsed.model_dump()

In [14]:
df = pd.read_csv(r'C:\Users\CoreX\OneDrive\Desktop\CareLoop_Clinics\Data\Followup_notes.csv')
df.head()
df = df.dropna(subset='note_text')

In [15]:
DEFAULT = {
    'root_cause' :"other",
    'confidence' : 0.0,
    'summary' : "No note recorded",
    'recommended_action' : "Re-call patient"
}

for idx, row in df.iterrows():
    if pd.isna(row['note_text']):
        result = {**DEFAULT, 'validation_status' : "skipped_blank" }
    elif row['note_text'].strip() == '':
        result = {**DEFAULT, 'validation_status' : "skipped_blank" }
    else: 
        try: 
            result = {**analyse_note(row['note_text']), 'validation_status' : "valid" }
        except Exception as e:
            result = {**DEFAULT, 'validation_status' : "invalid"}
            result['summary'] = f"Error occured {e}"

    for key, val in result.items():
        df.loc[idx, key] = val

In [19]:
df.to_csv("../Data/analysed_notes.csv")